# Lab 5

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/taipeitechmmslab/MMSLAB-Pytorch/blob/master/Lab5.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/taipeitechmmslab/MMSLAB-Pytorch/blob/master/Lab5.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

### Import

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


## 權重初始化

定義全連接網路架構

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# 定義網路架構
class DenseNetwork(nn.Module):
  def __init__(self, act='sigmoid'):
    super().__init__()
    self.layers = nn.ModuleList()
    for i in range(5):
      # 每層依序加入 Linear layer與activation function
      self.layers.append(nn.Linear(100, 100, bias=False, device=device))
      if act == 'sigmoid':
        self.layers.append(nn.Sigmoid())
      elif act == 'relu':
        self.layers.append(nn.ReLU())
    # 初始化和self.layer一樣長度的空陣列來儲存輸出
    self.layer_output = [None for _ in range(len(self.layers))]

  def forward(self, x):
    for layer in self.layers:
      x = layer(x)
    return x


定義forward hook和協助視覺化的程式

In [ ]:
# forward hook綁定net與index，如此每次forward時都會更新net.layer_output內的數值
def forward_hook(net, index):
  def hook(model, input, output):
    # 由於我們只是要觀察輸出，不需要梯度計算，這邊額外的把輸出從計算圖 detach, 再送入 cpu 並轉換成 nupy 來避免佔用 GPU 記憶體
    net.layer_output[index] = output.detach().cpu().numpy()
  return hook

# vshow_hist將輸入的資料輸出成可視化的直方圖
def vshow_hist(in_list):
  len_l = len(in_list)
  plt.figure(figsize=(len_l * 2, 5))
  for i, layer_output in enumerate(in_list):
    plt.subplot(1, len(in_list), i + 1) 
    plt.title(str(i+1) + "-layer")
    if i != 0:
        plt.yticks([], [])
    plt.hist(layer_output.flatten(), 30, range=[0,1])
  plt.show()



1. RandomNormal (std 1)

In [ ]:
# 定義權重初始化方法
def init_normal(m):
  if isinstance(m, nn.Linear):
    torch.nn.init.normal_(m.weight, std=1)

# 初始化網路架構
normal_net = DenseNetwork(act='sigmoid')
# 將權重初始化方法套用到網路中
normal_net.apply(init_normal)  
# 將 forward hook 加入到每一層中
for i in range(len(normal_net.layers)):
  normal_net.layers[i].register_forward_hook(forward_hook(normal_net, i))
# 將網路轉換到 GPU 上
normal_net.to(device)

# 輸入資料為 (100, 100)的亂數
in_data = np.random.randn(100, 100).astype(np.float32)
in_data = torch.tensor(in_data)
in_data = in_data.to(device)

# 產生輸出
output = normal_net(in_data)
# 這邊輸入normal_net.layer_output[1::2]來可視化activation function的輸出
# [1::2] 代表從 1 開始, 每次 +2, 也就是只抓取 nn.ModuleList 中 Sigmoid 的輸出
vshow_hist(normal_net.layer_output[1::2])


2. RandomNormal (std 0.01)

In [ ]:
# 定義權重初始化方法
def init_normal(m):
  if isinstance(m, nn.Linear):
    torch.nn.init.normal_(m.weight, std=0.01)
normal_net = DenseNetwork(act='sigmoid')  # 建立網路架構
normal_net.apply(init_normal)   # 將權重初始化方法套用到網路中
# 將forward hook加入到每一層中
for i in range(len(normal_net.layers)):
  normal_net.layers[i].register_forward_hook(forward_hook(normal_net, i))
normal_net.to(device)
# 產生輸出
output = normal_net(in_data)
# 這邊輸入normal_net.layer_output[1::2]來可視化activation function的輸出
vshow_hist(normal_net.layer_output[1::2])


3. Xavier/Glorot Initialization (Sigmoid)

In [ ]:
# 定義權重初始化方法
def init_xavier(m):
  if isinstance(m, nn.Linear):
    torch.nn.init.xavier_uniform_(m.weight)
xavier_net = DenseNetwork()
xavier_net.apply(init_xavier)
# 套用forward hook
for i in range(len(xavier_net.layers)):
  xavier_net.layers[i].register_forward_hook(forward_hook(xavier_net, i))
xavier_net.to(device)
# 可視化結果
output = xavier_net(in_data)
vshow_hist(xavier_net.layer_output[1::2])


4. Xavier/Glorot Initialization (ReLU)

In [ ]:
xavier_relu_net = DenseNetwork(act='relu')  # 將act參數改為relu
xavier_relu_net.apply(init_xavier)

for i in range(len(xavier_relu_net.layers)):
  xavier_relu_net.layers[i].register_forward_hook(forward_hook(xavier_relu_net, i))
xavier_relu_net.to(device)

output = xavier_relu_net(in_data)
vshow_hist(xavier_relu_net.layer_output[1::2])


5. He initialization

In [ ]:
# He init with ReLU
def init_he(m):
  if isinstance(m, nn.Linear):
    torch.nn.init.kaiming_uniform_(m.weight)
he_relu_net = DenseNetwork(act='relu')
he_relu_net.apply(init_he)

for i in range(len(he_relu_net.layers)):
  he_relu_net.layers[i].register_forward_hook(forward_hook(he_relu_net, i))
he_relu_net.to(device)

output = he_relu_net(in_data)
vshow_hist(he_relu_net.layer_output[1::2])


# 實驗一：使用CIFAR-10數據集實驗證三種權重初始化方法和Batch Normalization之比較

### Import必要套件和初始化計算裝置

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torchinfo import summary
from tqdm import tqdm
from torchvision import transforms as T
import torchvision
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

# 初始化計算裝置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### 定義影像增強與訓練相關函數

資料增強轉換、計算準確度、評估以及訓練的函式

In [ ]:
def image_transform_loader(img_size, with_aug=False, p=.5, flip_h=True, flip_v=False, color=False, contrast=False, sharpness=False, crop_rand=False, crop_center=False, blur=False, rotate=False):
    transform_list = [T.ToTensor()]
    if with_aug:
        if flip_h:
            transform_list += [T.RandomHorizontalFlip(p)]
        if flip_v:
            transform_list += [T.RandomVerticalFlip(p)]
        if color:
            transform_list += [T.ColorJitter(brightness=.5, hue=.3)]
        if contrast:
            transform_list += [T.RandomAutocontrast(p)]
        if sharpness:
            transform_list += [T.RandomAdjustSharpness(sharpness_factor=2, p=p)]
        if blur:
            transform_list += [T.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5))]
        if crop_rand:
            to_size = int(img_size * .8)
            transform_list += [T.RandomCrop(size=(to_size, to_size))]
        if crop_center:
            transform_list += [T.CenterCrop(size=img_size)]
        if rotate:
            transform_list += [T.RandomRotation(degrees=5)]
            
    transform_list += [T.Resize(size=img_size)]
    transform_list += [T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
    return T.Compose(transform_list)


def metric(x, y):
    """ 計算預測結果的 accuracy """
    x = torch.argmax(x, 1)
    accu = torch.sum(x == y) / y.shape[0] * 100
    return accu


def compute_loss_and_accuracy(batch, net, loss_func, device=None):
    """ 計算一個 batch 的 loss 和 accuracy """
    x, y = batch
    x = x.to(device)
    y = y.to(device)
    outputs = net(x)
    return loss_func(outputs, y), metric(outputs, y)


def evaluate(net, data_loader, loss_function, device):
    """ 對整個網路進行 validation set 的準確度判定 """
    net.eval()
    loss_total, accu_total = 0, 0
    for batch in data_loader:
        with torch.no_grad():
            loss_batch, accu_batch = compute_loss_and_accuracy(batch, net, loss_function, device)
            loss_total += loss_batch.item()
            accu_total += accu_batch.item()

    loss = loss_total / len(data_loader)
    accu = accu_total / len(data_loader)
    return loss, accu


def fit(epochs, lr, net, train_loader, val_loader, writer, opt_func=torch.optim.AdamW, device=None):
    """ 對網路進行訓練和測試, 並藉由 tensorboard writer 進行紀錄 """
    optimizer = opt_func(net.parameters(), lr)
    loss_func = nn.CrossEntropyLoss()
    
    for epoch in tqdm(range(epochs), desc='Epochs'):
        loss_train = 0
        train_accu_total = 0
        net.train()
        
        for batch in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}', leave=False):
            loss, accu_train = compute_loss_and_accuracy(batch, net, loss_func, device)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            loss_train += loss.item()
            train_accu_total += accu_train.item()
            
        loss_val, accu_val = evaluate(net, val_loader, loss_func, device)

        if epoch % 10 == 0:
            print(f'[{epoch}/{epochs}] Training loss: {loss_train:.4f}, validation loss: {loss_val:.4f}')

        # tensorboard 相關紀錄
        writer.add_scalar('loss/train', loss_train / len(train_loader), epoch)
        writer.add_scalar('loss/valid', loss_val, epoch)
        writer.add_scalar('accu/train', train_accu_total / len(train_loader), epoch)
        writer.add_scalar('accu/valid', accu_val, epoch)
        
        for index, t in enumerate(net.named_parameters()):
            name, param = t
            writer.add_histogram(name, param, epoch)

建立網路架構

In [ ]:
class ConvNet(nn.Module):
    """ 卷積神經網路, 可以選擇是否使用 batch normalization """
    def __init__(self, use_bn=False):
        super().__init__()
        place_holder = lambda x: nn.BatchNorm2d(x) if use_bn else nn.Identity()
        place_holder_1d = lambda x: nn.BatchNorm1d(x) if use_bn else nn.Identity()
        
        self.layers = nn.ModuleList([
            nn.Conv2d(3, 64, (3, 3)),
            place_holder(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 2), (2, 2)),
            nn.Conv2d(64, 128, (3, 3)),
            place_holder(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, (3, 3)),
            place_holder(256),
            nn.ReLU(),
            nn.Conv2d(256, 128, (3, 3)),
            place_holder(128),
            nn.ReLU(),
            nn.Conv2d(128, 64, (3, 3)),
            place_holder(64),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 64),
            place_holder_1d(64),
            nn.ReLU(),
            nn.Dropout(.5),
            nn.Linear(64, 10, bias=False),
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


# 權重初始化
def init_weights(method='xavier'):
    def init(m):
        if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
            if method == 'xavier':
                torch.nn.init.xavier_normal_(m.weight)
            elif method == 'he':
                torch.nn.init.kaiming_normal_(m.weight)
            elif method == 'normal':  
                torch.nn.init.normal_(m.weight, std=1)
    return init

### 讀取數據
載入CIFAR-10資料集並切分為訓練集與驗證集。

In [ ]:
batch_size = 128
cpu_num = 4 if os.cpu_count() > 4 else os.cpu_count()
if os.name == 'nt':
    cpu_num = 0

img_size = 28
transform = image_transform_loader(img_size)
transform_aug = image_transform_loader(img_size, with_aug=True, rotate=True, flip_v=True, contrast=True, sharpness=True)

dataset = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
dataset_aug = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform_aug, download=True)

d_len = len(dataset)
np.random.seed(9527)
indices = np.arange(d_len)
np.random.shuffle(indices)
np.random.seed()

train_indices = indices[:int(d_len * .7)]
valid_indices = indices[int(d_len * .7):]

train_subset = torch.utils.data.Subset(dataset_aug, train_indices)
valid_subset = torch.utils.data.Subset(dataset, valid_indices)

loader_train = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=cpu_num, pin_memory=True)
loader_valid = DataLoader(valid_subset, batch_size=batch_size, shuffle=False, num_workers=cpu_num, pin_memory=True)

# 將一組 batch 的資料取出, 供後續 tensorboard 進行網路架構視覺化使用
dataiter = iter(loader_train)
images, labels = next(dataiter)

### 執行訓練迴圈

In [ ]:
# Tensorboard 相關初始化
model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)

# 訓練相關超參數
epochs = 50
lr = 1e-3

# 訓練迴圈相關參數
model_seq = zip(
    ('xavier', 'he', 'normal', 'xavier'),  # 權重初始化方法
    (True, False, False, False),           # 是否使用 batch normalization
    ('model_bn', 'model_he', 'model_normal', 'model_xavier')  # 模型名稱
)

for init_method, use_bn, name in model_seq:
    model = ConvNet(use_bn=use_bn).to(device)
    
    model.apply(init_weights(init_method))
    writer = SummaryWriter(os.path.join(model_dir, name))
    
    fit(epochs, lr, model, loader_train, loader_valid, writer, device=device)
    
    # 使用 tensorboard 紀錄模型架構
    model.eval()
    writer.add_graph(model, torch.rand_like(images, device=device))
    writer.close()

開啟TensorBoard查看訓練記錄

In [ ]:
%load_ext tensorboard
%tensorboard --logdir models